# CodeSwitch-TTS — text/confidence residual (v7), unchanged 5% gate

**This notebook is the honest continuation of the v6 real-speech run.** It does not
retrain the draft, does not change the metric, and does not lower the gate.

## The truth so far

| Run | `p` (draft CB0) | `c` (CB1–8 \| CB0 right) | `w` (CB1–8 \| CB0 wrong) | `rt_a` | `rd_a` | Verdict |
|---|---:|---:|---:|---:|---:|---|
| 200-row pilot | — | — | — | 2.6133% | 1.7593% | **FAIL** |
| 6 h real Rasa | 7.9383% | 8.8726% | 1.6771% | 8.8726% | 2.2487% | **FAIL** |
| 6 h + LR floor `1e-5` | 7.9383% | 10.6244% | 1.5105% | 10.6244% | 2.2340% | **FAIL** |

> The `p/c/w` values above are the recorded ledger. Section 6 recomputes them
> exactly from the preserved checkpoint; the recomputed values supersede this
> table for any decision.

The gate is exactly

```text
rd_a = p * c + (1 - p) * w
```

At the best measured `c = 10.6244%` and `w = 1.5105%`, the draft would need
**`p ≈ 38.3%`** to pass. At the measured `p = 7.9383%` and `c = 10.6244%`, the
wrong-CB0 residual would need **`w ≈ 4.52%`** to pass. The current `rd_a` is
~22.9× chance; 5% is ~51.2× chance.

**Why this notebook exists.** The binding term is `w`, not `c`: ~92% of deployed
frames have a wrong draft CB0. A CB0-only residual cannot recover those frames.
This notebook makes one pre-registered, one-variable change: the residual can
see the draft's **frame-level text features** and its **CB0 confidence**, and
learn a fusion gate. On a wrong CB0 it can fall back to text; on a confident
CB0 it can trust the token. Nothing else changes.

## Stop rules

- The gate is still `rt_a >= 5%` **and** `rd_a >= 5%` from the unchanged
  `diagnose_audio_bug.py`.
- If it fails, that is the result. Do not claim B/C quality, do not run the
  60-utterance evaluation, and do not present a re-scope as a pass.
- A gate re-scope or a new acceptance metric needs **explicit user sign-off**.
- The prior W&B evidence is imported by ID, never rewritten:
  pilot data `gd4ea2go`, pilot train `ucsyx8e0`, pilot gate `qg3pnhk5`;
  6 h prep `ck8i90t6`, draft `92gb8bnm`, gate `c3w0vxxy`.


## 1. Configuration

Everything is explicit. Paths are searched in order; if the 6 h artifact or
cache is not mounted, set the W&B/HF pull variables. No secret is written to
disk except the temporary Git askpass helper, which is removed after cloning.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/tripathiji1312/codeSwitch.git"
REPO_REF = "fix/013-bc-noise"          # must contain the text/confidence residual
REPO_DIR = Path("/kaggle/working/codeSwitch")

DATASET_CHOICE = "rasa_ta"
CKPT_TAG = "v7_textres"
CKPT_DIR = REPO_DIR / f"codeswitch_ckpt_{CKPT_TAG}"
DIAG_DIR = REPO_DIR / f"eval_results_{CKPT_TAG}_diagnostics"
BASE_GATE_DIR = REPO_DIR / f"eval_results_{CKPT_TAG}_base_gate"
BASE_CEIL_DIR = REPO_DIR / f"eval_results_{CKPT_TAG}_base_ceiling"
NEW_GATE_DIR = REPO_DIR / f"eval_results_{CKPT_TAG}_gate"
NEW_CEIL_DIR = REPO_DIR / f"eval_results_{CKPT_TAG}_ceiling"
EVAL_DIR = REPO_DIR / f"eval_results_{CKPT_TAG}_eval60"
EVAL_AUDIO = REPO_DIR / f"eval_audio_{CKPT_TAG}_eval60"

# Where the preserved 6 h run may live. The first directory with draft_cb0.pt wins.
BASE_CKPT_CANDIDATES = [
    Path("/kaggle/working/codeSwitch/codeswitch_ckpt_v6_6h"),
    Path("/kaggle/input/codeswitch-ckpt-v6-6h"),
    Path("/kaggle/input/codeswitch-v6-6h/codeswitch_ckpt_v6_6h"),
    Path("/kaggle/working/codeswitch_ckpt_v6_6h"),
]
BASE_CACHE_CANDIDATES = [
    Path("/kaggle/working/codeSwitch/real_cache/rasa_ta"),
    Path("/kaggle/input/codeswitch-real-cache/rasa_ta"),
    Path("/kaggle/input/codeswitch-v6-6h/real_cache/rasa_ta"),
    Path("/kaggle/working/real_cache/rasa_ta"),
]

# Optional pulls when the preserved files are not mounted.
WANDB_PROJECT = "codeswitch-tts"
WANDB_ENTITY = None                    # str or None
WANDB_GROUP = "v7-textres"
BASE_CKPT_ARTIFACT = f"{WANDB_PROJECT}/codeswitch-v6-rasa_ta:latest"  # "" disables the W&B pull
CACHE_ARTIFACT = ""                    # e.g. "real-speech-cache-rasa_ta:latest"
HF_CHECKPOINT_REPO = ""                # e.g. "user/codeswitch-v6-6h" (model repo)
HF_CACHE_REPO = ""                     # e.g. "user/codeswitch-real-cache-rasa_ta" (dataset)
HF_PUSH_REPO = ""                      # optional model repo for the new checkpoint

# Run switches.
RUN_DIAGNOSTICS = True
REPRODUCE_BASE_GATE = True
RUN_TRAIN = True
RUN_GATE = True
RUN_EVAL = False
ALLOW_EVAL_WITHOUT_GATE = False
ROBUST_GATE_SHARDS = 20                # 0 disables the larger-sample recheck
NGRAM_SHARDS = 200                     # real-cache n-gram calibration
ALLOW_SMALL_CACHE = False

# Residual intervention. One variable: text/confidence conditioning. Artificial
# CB0 corruption is OFF by default; natural draft errors already expose it.
RESIDUAL_TEXT = True
RESIDUAL_CONFIDENCE = True
CB0_DROPOUT = 0.0
CB0_CORRUPTION = 0.0
RESIDUAL_EPOCHS = 150
RESIDUAL_RAMP_EPOCHS = 100
RESIDUAL_MIN_EPOCHS = 150
RESIDUAL_D_MODEL = 512
RESIDUAL_LAYERS = 6
RESIDUAL_BATCH = 32
RESIDUAL_LR = 3e-4
MAX_MINUTES_RESIDUAL = 90.0

GATE_SHARDS = 4
ASR_MODEL = "openai/whisper-large-v3"
SEED = 42
LOG_ARTIFACTS = True
DO_ZIP = True

PRIOR_RUNS = {
    "pilot_data": "gd4ea2go",
    "pilot_train": "ucsyx8e0",
    "pilot_gate": "qg3pnhk5",
    "six_hour_prep": "ck8i90t6",
    "six_hour_draft": "92gb8bnm",
    "six_hour_gate": "c3w0vxxy",
    "six_hour_lr_floor_residual": "c3w0vxxy",
}
PRIOR_TRUTH = {
    "pilot": {"rt_a": 2.6132509690, "rd_a": 1.7593265504},
    "six_hour": {"p": 7.9383, "c": 8.8726, "w": 1.6771, "rt_a": 8.8726, "rd_a": 2.2487142218},
    "lr_floor": {"p": 7.9383, "c": 10.6244, "w": 1.5105, "rt_a": 10.6244409660, "rd_a": 2.2340395796},
}

STATE = {
    "commit": None,
    "base_ckpt": None,
    "cache": None,
    "base_meta": {},
    "new_meta": {},
    "base_gate": None,
    "base_ceiling": None,
    "new_gate": None,
    "new_ceiling": None,
    "gate_pass": None,
    "wandb_urls": [],
}

print("repo:", REPO_DIR)
print("base checkpoint candidates:", *BASE_CKPT_CANDIDATES, sep="\n  ")
print("cache candidates:", *BASE_CACHE_CANDIDATES, sep="\n  ")
print("new checkpoint:", CKPT_DIR)
print("switches:", dict(RUN_DIAGNOSTICS=RUN_DIAGNOSTICS, RUN_TRAIN=RUN_TRAIN,
                        RUN_GATE=RUN_GATE, RUN_EVAL=RUN_EVAL))


## 2. Secrets and helpers

`HF_TOKEN`, `WANDB_API_KEY`, and `GITHUB_TOKEN` are read from Kaggle Secrets
first and environment variables second. Values are never printed. If W&B is
unavailable the notebook runs offline; artifact pulls then fall back to HF or a
mounted path.


In [ ]:
import json
import os
import re
import shlex
import subprocess
import sys
import zipfile


def _secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
        if value:
            return value
    except Exception as exc:
        print(f"[secrets] {name} not in Kaggle Secrets ({type(exc).__name__})")
    return os.environ.get(name)


for _name in ("WANDB_API_KEY", "HF_TOKEN", "GITHUB_TOKEN"):
    _value = _secret(_name)
    if _value:
        os.environ[_name] = _value

print("WANDB_API_KEY:", "set (hidden)" if os.environ.get("WANDB_API_KEY") else "missing -> offline")
print("HF_TOKEN:", "set (hidden)" if os.environ.get("HF_TOKEN") else "missing")
print("GITHUB_TOKEN:", "set (hidden)" if os.environ.get("GITHUB_TOKEN") else "missing")
if not os.environ.get("WANDB_API_KEY"):
    os.environ.setdefault("WANDB_MODE", "offline")

PYTHON = sys.executable


def run(cmd, cwd=None, check=True, env=None, capture=False):
    cmd = [str(part) for part in cmd]
    print("$ " + " ".join(shlex.quote(part) for part in cmd))
    kwargs = dict(cwd=str(cwd) if cwd is not None else None,
                  env=env if env is not None else os.environ.copy(),
                  text=True)
    if capture:
        kwargs["stdout"] = subprocess.PIPE
        kwargs["stderr"] = subprocess.STDOUT
    proc = subprocess.run(cmd, **kwargs)
    if check and proc.returncode != 0:
        if capture and proc.stdout:
            print(proc.stdout[-8000:])
        raise RuntimeError(f"command failed ({proc.returncode}): {' '.join(cmd)}")
    return proc


def run_stream(cmd, cwd, run_obj=None, stage="stage"):
    """Run a long command, print output, and log residual epochs to W&B."""
    cmd = [str(part) for part in cmd]
    print("$ " + " ".join(shlex.quote(part) for part in cmd))
    stream_env = os.environ.copy()
    stream_env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.Popen(cmd, cwd=str(cwd), env=stream_env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        if run_obj is None:
            continue
        match = re.search(
            r"\[Residual\] epoch\s+(\d+)\s+\|\s+train\s+([0-9.]+)\s+\(sched\s+([0-9.]+)\)"
            r"\s+\|\s+val\s+([0-9.]+).*?lr\s+([0-9.eE+-]+)",
            line,
        )
        if match:
            run_obj.log({
                "residual/epoch": int(match.group(1)),
                "residual/train_loss": float(match.group(2)),
                "residual/sched_p": float(match.group(3)),
                "residual/val_loss": float(match.group(4)),
                "residual/lr": float(match.group(5)),
            })
        match = re.search(r"\[Draft\] epoch\s+(\d+).*?train\s+([0-9.]+).*?val\s+([0-9.]+)", line)
        if match:
            run_obj.log({
                "draft/epoch": int(match.group(1)),
                "draft/train_loss": float(match.group(2)),
                "draft/val_loss": float(match.group(3)),
            })
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"{stage} failed with exit code {code}")
    return code


print("PYTHON:", PYTHON)


## 3. Clone / pull the repo

The branch must be pushed before Kaggle can see it. This cell prints the exact
commit and refuses to continue if the text/confidence residual flags are absent.
The temporary askpass helper is deleted after the clone/fetch.


In [ ]:
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
git_env = os.environ.copy()
askpass_path = None
if os.environ.get("GITHUB_TOKEN"):
    askpass_path = Path("/tmp/codeswitch-git-askpass.sh")
    askpass_path.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *Password*) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
        'esac\n',
        encoding="utf-8",
    )
    askpass_path.chmod(0o700)
    git_env["GIT_ASKPASS"] = str(askpass_path)
    git_env["GIT_TERMINAL_PROMPT"] = "0"

try:
    if (REPO_DIR / ".git").is_dir():
        print("repo exists; fetching", REPO_REF)
        fetch = run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR,
                    check=False, env=git_env, capture=True)
        if fetch.returncode != 0:
            print(fetch.stdout or "")
            raise RuntimeError(
                f"Could not fetch origin/{REPO_REF}. Push the branch first, or set "
                "REPO_REF='main' after merging. See git output above."
            )
        run(["git", "checkout", "-B", REPO_REF, "FETCH_HEAD"], cwd=REPO_DIR, env=git_env)
    else:
        if REPO_DIR.exists():
            raise RuntimeError(f"{REPO_DIR} exists but is not a git checkout; move it away.")
        try:
            run(["git", "clone", "--branch", REPO_REF, "--single-branch",
                 REPO_URL, str(REPO_DIR)], cwd=REPO_DIR.parent, env=git_env)
        except Exception as exc:
            raise RuntimeError(
                f"Could not clone origin/{REPO_REF}. Push the branch first, or "
                "set REPO_REF='main' after merging. Original error: "
                f"{exc}"
            ) from exc
finally:
    if askpass_path is not None:
        askpass_path.unlink(missing_ok=True)

for required in ("prepare_real_speech.py", "train_draft_distill.py",
                 "diagnose_audio_bug.py", "diagnose_gate_ceiling.py"):
    if not (REPO_DIR / required).is_file():
        raise RuntimeError(f"checkout is missing {required}")

sys.path.insert(0, str(REPO_DIR))
STATE["commit"] = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()
status = subprocess.check_output(
    ["git", "status", "--short"], cwd=REPO_DIR, text=True
).strip()
train_help = subprocess.check_output(
    [PYTHON, "train_draft_distill.py", "--help"], cwd=REPO_DIR, text=True
)
for flag in ("--residual-text-conditioning", "--residual-confidence-conditioning",
             "--residual-cb0-dropout", "--residual-cb0-corruption"):
    if flag not in train_help:
        raise RuntimeError(
            f"pushed branch is stale: {flag} is missing. Commit and push the "
            "text/confidence residual changes, then re-run this cell."
        )
if "--draft-frame-decoder-layers" not in train_help:
    raise RuntimeError("pushed branch is stale: no frame-decoder flag")
print("commit:", STATE["commit"])
print("working tree:", "clean" if not status else status)


## 4. Locate the preserved 6 h checkpoint and cache

Order: mounted path → W&B artifact → Hugging Face snapshot. The cache is the
same 4,015-shard / 6.0001 h Rasa slice. A smaller cache is rejected by default
because its gate numbers are not comparable to the 6 h ledger.


In [ ]:
import torch


def _has_ckpt(path):
    p = Path(path)
    return p.is_dir() and (p / "draft_cb0.pt").is_file()


def _find_mounted_ckpt():
    for candidate in BASE_CKPT_CANDIDATES:
        if _has_ckpt(candidate):
            return candidate
    for base in (Path("/kaggle/input"), Path("/kaggle/working")):
        if not base.is_dir():
            continue
        for path in base.rglob("draft_cb0.pt"):
            parent = path.parent
            if CKPT_TAG in str(parent):
                continue  # never mistake this run's output for the base checkpoint
            if (parent / "meta.json").is_file() or (parent / "residual_cb1_8.pt").is_file():
                return parent
    return None


def _cache_size(path):
    return len(list(Path(path).glob("shard_*.npz")))


def _find_mounted_cache():
    for candidate in BASE_CACHE_CANDIDATES:
        if candidate.is_dir() and _cache_size(candidate):
            return candidate
    fallbacks = []
    for base in (Path("/kaggle/input"), Path("/kaggle/working")):
        if not base.is_dir():
            continue
        for path in base.rglob("shard_*.npz"):
            parent = path.parent
            if parent in fallbacks:
                continue
            fallbacks.append(parent)
            if _cache_size(parent) >= 1000:
                return parent
    return fallbacks[0] if fallbacks else None


def _artifact_ref(name):
    """Expand a W&B artifact shorthand to entity/project/name:alias."""
    if not name:
        return ""
    if name.count("/") >= 2:
        return name
    import wandb
    api = wandb.Api(timeout=60)
    entity = WANDB_ENTITY or api.default_entity
    if not entity:
        raise RuntimeError("W&B entity unknown; set WANDB_ENTITY")
    if name.count("/") == 1:
        return f"{entity}/{name}"
    return f"{entity}/{WANDB_PROJECT}/{name}"


def _pull_wandb_ckpt():
    if not BASE_CKPT_ARTIFACT:
        return None
    try:
        import wandb
        api = wandb.Api(timeout=60)
        artifact = api.artifact(_artifact_ref(BASE_CKPT_ARTIFACT))
        downloaded = Path(artifact.download())
        for path in [downloaded, *downloaded.rglob("draft_cb0.pt")]:
            parent = path if path.is_dir() else path.parent
            if _has_ckpt(parent):
                return parent
        print("W&B artifact downloaded but no draft_cb0.pt found:", downloaded)
    except Exception as exc:
        print(f"[W&B pull] {type(exc).__name__}: {exc}")
    return None


def _pull_hf_checkpoint():
    if not HF_CHECKPOINT_REPO:
        return None
    try:
        from huggingface_hub import snapshot_download
        target = Path("/kaggle/working/pulled_v6_checkpoint")
        snapshot_download(repo_id=HF_CHECKPOINT_REPO, repo_type="model",
                          local_dir=target, token=os.environ.get("HF_TOKEN"))
        for path in [target, *target.rglob("draft_cb0.pt")]:
            parent = path if path.is_dir() else path.parent
            if _has_ckpt(parent):
                return parent
    except Exception as exc:
        print(f"[HF pull] {type(exc).__name__}: {exc}")
    return None


def _pull_wandb_cache():
    if not CACHE_ARTIFACT:
        return None
    try:
        import wandb
        api = wandb.Api(timeout=60)
        downloaded = Path(api.artifact(_artifact_ref(CACHE_ARTIFACT)).download())
        if any(downloaded.rglob("shard_*.npz")):
            for path in downloaded.rglob("shard_*.npz"):
                return path.parent
    except Exception as exc:
        print(f"[W&B cache pull] {type(exc).__name__}: {exc}")
    return None


def _pull_hf_cache():
    if not HF_CACHE_REPO:
        return None
    try:
        from huggingface_hub import snapshot_download
        target = Path("/kaggle/working/pulled_real_cache/rasa_ta")
        snapshot_download(repo_id=HF_CACHE_REPO, repo_type="dataset",
                          local_dir=target, token=os.environ.get("HF_TOKEN"))
        if any(target.rglob("shard_*.npz")):
            return next(target.rglob("shard_*.npz")).parent
    except Exception as exc:
        print(f"[HF cache pull] {type(exc).__name__}: {exc}")
    return None


base_ckpt = _find_mounted_ckpt() or _pull_wandb_ckpt() or _pull_hf_checkpoint()
cache_dir = _find_mounted_cache() or _pull_wandb_cache() or _pull_hf_cache()
if base_ckpt is None:
    raise RuntimeError(
        "No v6 6 h draft checkpoint found. Mount it at one of "
        f"{[str(p) for p in BASE_CKPT_CANDIDATES]}, set BASE_CKPT_ARTIFACT, "
        "or set HF_CHECKPOINT_REPO."
    )
if cache_dir is None:
    raise RuntimeError(
        "No real-speech shard cache found. Mount the 6 h cache, set "
        "CACHE_ARTIFACT/HF_CACHE_REPO, or re-run prepare_real_speech.py. "
        "Do not substitute the 10-shard FLEURS smoke for gate numbers."
    )

shards = sorted(Path(cache_dir).glob("shard_*.npz"))
STATE["base_ckpt"] = Path(base_ckpt)
STATE["cache"] = Path(cache_dir)
EXPECTED_CACHE_SHARDS = 4015
if len(shards) != EXPECTED_CACHE_SHARDS:
    print(f"WARNING: cache has {len(shards)} shards; the 6 h ledger has "
          f"{EXPECTED_CACHE_SHARDS}. Gate numbers are not directly comparable "
          "unless this is the same cache.")
if not ALLOW_SMALL_CACHE and len(shards) < 1000:
    raise RuntimeError(
        f"cache {cache_dir} has only {len(shards)} shards; the 6 h ledger has "
        "4,015. Its gate numbers are not comparable. Set ALLOW_SMALL_CACHE=True "
        "only for a plumbing smoke, and do not report it as the 6 h result."
    )
print("base checkpoint:", STATE["base_ckpt"])
print("shards:", len(shards), "| cache:", STATE["cache"])


## 5. Audit the base checkpoint before spending GPU time

This is the compatibility gate for the intervention: the draft must expose
`frame_features` (a frame decoder), and the old residual must be loadable. The
cell also prints the exact old checkpoint metadata so the notebook is
self-contained.


In [ ]:
import dataclasses
from codeswitch_tts.config import DraftModelConfig, ResidualModelConfig
from codeswitch_tts.models.draft_transformer import DraftTransformer
from codeswitch_tts.models.residual_decoder import ResidualDecoder

base_ckpt = STATE["base_ckpt"]
cache_dir = STATE["cache"]
meta_path = base_ckpt / "meta.json"
base_meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.is_file() else {}
STATE["base_meta"] = base_meta

draft_payload = torch.load(base_ckpt / "draft_cb0.pt", map_location="cpu", weights_only=False)
draft_cfg = DraftModelConfig(**draft_payload.get("config", {}))
draft = DraftTransformer(draft_cfg).eval()
draft.load_state_dict({k.removeprefix("module."): v for k, v in draft_payload["state_dict"].items()})

residual_path = base_ckpt / "residual_cb1_8.pt"
residual_cfg = None
if residual_path.is_file():
    residual_payload = torch.load(residual_path, map_location="cpu", weights_only=False)
    residual_cfg = ResidualModelConfig(**residual_payload.get("config", {}))

print("draft config:", dataclasses.asdict(draft_cfg))
print("draft params:", f"{sum(p.numel() for p in draft.parameters()) / 1e6:.2f}M")
print("residual config:", dataclasses.asdict(residual_cfg) if residual_cfg else "missing")
print("base corpus_hours:", base_meta.get("corpus_hours"))
print("base duration_shards:", base_meta.get("duration_shards"),
      "/", base_meta.get("corpus_size"))
print("base draft stats:", {k: base_meta.get("draft", {}).get(k) for k in
      ("val_top1", "val_top5", "val_top1_gt_durations", "best_epoch")})
print("base residual stats:", {k: base_meta.get("residual", {}).get(k) for k in
      ("cb_top1_teacher_mean", "cb_top1_draft_mean", "best_epoch", "best_sched_p")})
if draft_cfg.frame_decoder_layers < 1:
    raise RuntimeError(
        "The preserved draft has no frame decoder, so it has no frame_features. "
        "Text conditioning cannot reuse it. Retrain the draft with "
        "--draft-frame-decoder-layers 1, or run a CB0-only residual."
    )

sample = next(iter(shards))
from train_draft_distill import TeacherShard
shard = TeacherShard.load(sample)
n = int(shard.frames)
with torch.inference_mode():
    out = draft.generate_sequence(text=shard.norm_text, num_frames=n, sample=False,
                                  device=torch.device("cpu"))
if out.frame_features is None:
    raise RuntimeError("draft returned no frame_features despite a frame decoder")
print("frame feature smoke:", tuple(out.frame_features.shape),
      "| draft CB0 tokens:", tuple(out.tokens.shape))

old_meta = {}
for label, key in (("six_hour", "six_hour"), ("lr_floor", "lr_floor")):
    old_meta[label] = PRIOR_TRUTH[key]
print("imported prior truth:", json.dumps(old_meta, indent=2))


## 6. Reproduce the old gate and measure the decomposition

Two diagnostics, both on the **unchanged** base checkpoint:

1. `diagnose_audio_bug.py` — the actual gate (`rt_a`, `rd_a`, 4 shards).
2. `diagnose_gate_ceiling.py` — conditional `p/c/w`, per-codebook table, top-k
   coverage, confidence deciles, and character-aligned text/CB0 n-gram
   baselines on real shards.

The diagnostic reports both the official per-shard `rd_a_gate_style` and the
frame-weighted `p/c/w` identity. Only the official number matches the gate
verdict; the conditional identity is not exact under equal-shard weighting.

The n-gram numbers are **empirical floors**, not neural ceilings. A text-only
n-gram reaching `w≈1%` means text alone explains little exact token identity;
it does not prove a neural text-conditioned residual cannot do better.


In [ ]:
import wandb

if os.environ.get("WANDB_API_KEY"):
    try:
        wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)
        print("W&B login: ok")
    except Exception as exc:
        print(f"W&B login failed ({type(exc).__name__}); offline")
        os.environ["WANDB_MODE"] = "offline"

run_kwargs = {"entity": WANDB_ENTITY} if WANDB_ENTITY else {}
diag_run = wandb.init(
    project=WANDB_PROJECT,
    job_type="v7-diagnostics",
    name=f"v7-diagnostics-{DATASET_CHOICE}",
    group=WANDB_GROUP,
    config={
        "git_commit": STATE["commit"],
        "base_checkpoint": str(STATE["base_ckpt"]),
        "cache": str(STATE["cache"]),
        "gate_shards": GATE_SHARDS,
        "ngram_shards": NGRAM_SHARDS,
        "prior_runs": PRIOR_RUNS,
        "prior_truth": PRIOR_TRUTH,
        "hypothesis": "text/confidence conditioning raises w on wrong-CB0 frames",
        "gate_unchanged": "rt_a >= 5 and rd_a >= 5",
    },
    reinit=True,
    **run_kwargs,
)

if RUN_DIAGNOSTICS:
    if REPRODUCE_BASE_GATE:
        base_gate_cmd = [
            PYTHON, "diagnose_audio_bug.py",
            "--ckpt", str(STATE["base_ckpt"]),
            "--teacher-cache", str(STATE["cache"]),
            "--shards", str(GATE_SHARDS),
            "--out", str(BASE_GATE_DIR),
            "--device", "cuda:0" if torch.cuda.is_available() else "cpu",
            "--no-audio",
        ]
        run(base_gate_cmd, cwd=REPO_DIR)
        base_report = json.loads((BASE_GATE_DIR / "diagnostic_report.json").read_text())
        STATE["base_gate"] = base_report["summary"]
        diag_run.log({f"base_gate/{k}": float(v) for k, v in base_report["summary"].items()})
        print("base gate:", base_report["summary"])

    ceiling_cmd = [
        PYTHON, "diagnose_gate_ceiling.py",
        "--ckpt", str(STATE["base_ckpt"]),
        "--teacher-cache", str(STATE["cache"]),
        "--shards", str(GATE_SHARDS),
        "--ngram-shards", str(NGRAM_SHARDS),
        "--out", str(BASE_CEIL_DIR),
        "--device", "cuda:0" if torch.cuda.is_available() else "cpu",
    ]
    run(ceiling_cmd, cwd=REPO_DIR)
    base_ceiling = json.loads((BASE_CEIL_DIR / "ceiling_report.json").read_text())
    STATE["base_ceiling"] = base_ceiling
    diag_run.log({
        "base/p": base_ceiling["gate"]["p"],
        "base/c": base_ceiling["gate"]["c"],
        "base/w": base_ceiling["gate"]["w"],
        "base/rd_a_from_terms": base_ceiling["gate"]["rd_a_from_terms"],
        "base/rd_a_gate_style": base_ceiling["gate"]["rd_a_gate_style"],
        "base/gate_style_identity_error": (
            base_ceiling["gate"]["gate_style_identity_error"]
        ),
        "base/required_p_at_current_c_w": (
            base_ceiling["gate"]["required_p_at_current_c_w"] or -1.0
        ),
        "base/required_w_at_current_p_c": (
            base_ceiling["gate"]["required_w_at_current_p_c"] or -1.0
        ),
    })
    print("base ceiling truth:", base_ceiling["truth"])
    if LOG_ARTIFACTS:
        art = wandb.Artifact(f"v7-base-diagnostics-{DATASET_CHOICE}", type="evaluation")
        if (BASE_CEIL_DIR / "ceiling_report.json").is_file():
            art.add_file(str(BASE_CEIL_DIR / "ceiling_report.json"))
        if (BASE_GATE_DIR / "diagnostic_report.json").is_file():
            art.add_file(str(BASE_GATE_DIR / "diagnostic_report.json"))
        diag_run.log_artifact(art)
else:
    print("RUN_DIAGNOSTICS=False; base diagnostics skipped")

if diag_run.url:
    STATE["wandb_urls"].append(diag_run.url)
print("W&B diagnostics:", diag_run.url or diag_run.dir)
diag_run.finish()


## 7. Train the text/confidence residual

This is a **residual-only** run against the preserved draft. The draft is copied
into the new checkpoint directory, never overwritten.

What changes:

- `--residual-text-conditioning`: the residual reads the draft's frame-level
  text features through a projection.
- `--residual-confidence-conditioning`: a learned fusion gate sees the draft's
  CB0 confidence and can down-weight an unreliable token.
- `--residual-sched 1.0`: unchanged from the 6 h baseline; the residual trains
  on the draft's natural CB0 errors.
- Artificial CB0 dropout/corruption stays **0** by default. That is a second,
  separate lever; use it only in a follow-up run.

Everything else matches the 6 h residual: 512-d, 6 blocks, batch 32, 150
epochs, 100-epoch ramp, 3e-4 LR, 90-minute budget.


In [ ]:
if RUN_TRAIN:
    if not RESIDUAL_TEXT:
        raise RuntimeError(
            "This notebook is the text-conditioning experiment; RESIDUAL_TEXT=False "
            "would repeat the failed CB0-only residual. Set it only for an ablation."
        )
    train_cmd = [
        PYTHON, "train_draft_distill.py",
        "--skip-teacher",
        "--skip-draft",
        "--draft-ckpt", str(STATE["base_ckpt"]),
        "--teacher-cache", str(STATE["cache"]),
        "--ckpt-dir", str(CKPT_DIR),
        "--device", "cuda:0" if torch.cuda.is_available() else "cpu",
        "--dtype", "fp16" if torch.cuda.is_available() else "fp32",
        "--seed", str(SEED),
        "--residual-epochs", str(RESIDUAL_EPOCHS),
        "--residual-batch", str(RESIDUAL_BATCH),
        "--residual-lr", str(RESIDUAL_LR),
        "--residual-ramp-epochs", str(RESIDUAL_RAMP_EPOCHS),
        "--residual-sched", "1.0",
        "--residual-d-model", str(RESIDUAL_D_MODEL),
        "--residual-layers", str(RESIDUAL_LAYERS),
        "--residual-text-conditioning",
        "--residual-cb0-dropout", str(CB0_DROPOUT),
        "--residual-cb0-corruption", str(CB0_CORRUPTION),
        "--min-epochs", str(RESIDUAL_MIN_EPOCHS),
        "--max-minutes-residual", str(MAX_MINUTES_RESIDUAL),
    ]
    if RESIDUAL_CONFIDENCE:
        train_cmd.append("--residual-confidence-conditioning")

    train_run = wandb.init(
        project=WANDB_PROJECT,
        job_type="v7-textres-train",
        name=f"v7-textres-{DATASET_CHOICE}",
        group=WANDB_GROUP,
        config={
            "git_commit": STATE["commit"],
            "base_checkpoint": str(STATE["base_ckpt"]),
            "cache": str(STATE["cache"]),
            "text_conditioning": True,
            "confidence_conditioning": bool(RESIDUAL_CONFIDENCE),
            "cb0_dropout": float(CB0_DROPOUT),
            "cb0_corruption": float(CB0_CORRUPTION),
            "epochs": RESIDUAL_EPOCHS,
            "ramp_epochs": RESIDUAL_RAMP_EPOCHS,
            "min_epochs": RESIDUAL_MIN_EPOCHS,
            "d_model": RESIDUAL_D_MODEL,
            "layers": RESIDUAL_LAYERS,
            "batch_size": RESIDUAL_BATCH,
            "lr": RESIDUAL_LR,
            "max_minutes": MAX_MINUTES_RESIDUAL,
            "hypothesis": "w rises above 4.52% at current p/c",
        },
        reinit=True,
        **run_kwargs,
    )
    try:
        run_stream(train_cmd, cwd=REPO_DIR, run_obj=train_run, stage="residual-training")
    except Exception:
        train_run.finish(exit_code=1)
        raise

    new_meta_path = CKPT_DIR / "meta.json"
    if not new_meta_path.is_file():
        train_run.finish(exit_code=1)
        raise RuntimeError(f"training wrote no meta.json: {new_meta_path}")
    STATE["new_meta"] = json.loads(new_meta_path.read_text(encoding="utf-8"))
    residual_stats = STATE["new_meta"].get("residual", {})
    for key in ("cb_top1_teacher_mean", "cb_top1_draft_mean", "cb_top1_text_mean",
                "best_epoch", "best_sched_p", "selected_full_draft", "epochs_run"):
        value = residual_stats.get(key)
        if isinstance(value, (int, float)):
            train_run.log({f"residual/{key}": float(value)})
    train_run.summary.update({
        "checkpoint": str(CKPT_DIR),
        "selected_full_draft": residual_stats.get("selected_full_draft"),
        "cb_top1_teacher_mean": residual_stats.get("cb_top1_teacher_mean"),
        "cb_top1_draft_mean": residual_stats.get("cb_top1_draft_mean"),
        "cb_top1_text_mean": residual_stats.get("cb_top1_text_mean"),
    })
    if LOG_ARTIFACTS:
        model_artifact = wandb.Artifact(
            f"codeswitch-v7-textres-{DATASET_CHOICE}", type="model"
        )
        for name in ("draft_cb0.pt", "residual_cb1_8.pt", "meta.json"):
            path = CKPT_DIR / name
            if path.is_file():
                model_artifact.add_file(str(path))
        train_run.log_artifact(model_artifact)
    if train_run.url:
        STATE["wandb_urls"].append(train_run.url)
    print("residual stats:", {k: residual_stats.get(k) for k in (
        "cb_top1_teacher_mean", "cb_top1_draft_mean", "cb_top1_text_mean",
        "best_epoch", "best_sched_p", "selected_full_draft")})
    print("W&B train:", train_run.url or train_run.dir)
    train_run.finish()
else:
    if (CKPT_DIR / "meta.json").is_file():
        STATE["new_meta"] = json.loads((CKPT_DIR / "meta.json").read_text())
    print("RUN_TRAIN=False; using existing checkpoint:", CKPT_DIR)


## 8. The unchanged gate on the new checkpoint

`diagnose_audio_bug.py` is executed with the same 4-shard sampling and the same
thresholds. `diagnose_gate_ceiling.py` then decomposes the result and reports
the official per-shard `rd_a_gate_style`, the frame-weighted identity, and the
gate-style identity error. A larger 20-shard recheck is reported as a robustness
diagnostic; it does **not** replace the 4-shard pass/fail.

The two numbers that matter:

- `rt_a >= 5.0%`
- `rd_a >= 5.0%`

If `rd_a` fails, the gate fails.


In [ ]:
if RUN_GATE:
    if not (CKPT_DIR / "residual_cb1_8.pt").is_file():
        raise RuntimeError("RUN_GATE=True requires a trained residual checkpoint")

    gate_run = wandb.init(
        project=WANDB_PROJECT,
        job_type="v7-gate",
        name=f"v7-gate-{DATASET_CHOICE}",
        group=WANDB_GROUP,
        config={
            "git_commit": STATE["commit"],
            "checkpoint": str(CKPT_DIR),
            "threshold_rt_a": 5.0,
            "threshold_rd_a": 5.0,
            "gate_shards": GATE_SHARDS,
            "robust_gate_shards": ROBUST_GATE_SHARDS,
            "base_gate": STATE.get("base_gate"),
            "base_ceiling_gate": (STATE.get("base_ceiling") or {}).get("gate"),
        },
        reinit=True,
        **run_kwargs,
    )

    gate_cmd = [
        PYTHON, "diagnose_audio_bug.py",
        "--ckpt", str(CKPT_DIR),
        "--teacher-cache", str(STATE["cache"]),
        "--shards", str(GATE_SHARDS),
        "--out", str(NEW_GATE_DIR),
        "--device", "cuda:0" if torch.cuda.is_available() else "cpu",
        "--no-audio",
    ]
    run(gate_cmd, cwd=REPO_DIR)
    gate_report = json.loads((NEW_GATE_DIR / "diagnostic_report.json").read_text())
    STATE["new_gate"] = gate_report["summary"]
    gate_summary = gate_report["summary"]
    STATE["gate_pass"] = bool(
        gate_summary["rt_a"] >= 5.0 and gate_summary["rd_a"] >= 5.0
    )
    gate_run.log({f"gate/{k}": float(v) for k, v in gate_summary.items()})
    gate_run.summary["gate_pass"] = STATE["gate_pass"]
    gate_run.summary["rt_a"] = gate_summary["rt_a"]
    gate_run.summary["rd_a"] = gate_summary["rd_a"]
    print("GATE", "PASS" if STATE["gate_pass"] else "FAIL", gate_summary)

    ceiling_cmd = [
        PYTHON, "diagnose_gate_ceiling.py",
        "--ckpt", str(CKPT_DIR),
        "--teacher-cache", str(STATE["cache"]),
        "--shards", str(GATE_SHARDS),
        "--skip-ngram",
        "--out", str(NEW_CEIL_DIR),
        "--device", "cuda:0" if torch.cuda.is_available() else "cpu",
    ]
    run(ceiling_cmd, cwd=REPO_DIR)
    new_ceiling = json.loads((NEW_CEIL_DIR / "ceiling_report.json").read_text())
    STATE["new_ceiling"] = new_ceiling
    gate = new_ceiling["gate"]
    gate_run.log({
        "decomposition/p": gate["p"],
        "decomposition/c": gate["c"],
        "decomposition/w": gate["w"],
        "decomposition/w_text_model": gate.get("w_text_model") or -1.0,
        "decomposition/rd_a_from_terms": gate["rd_a_from_terms"],
        "decomposition/rd_a_gate_style": gate["rd_a_gate_style"],
        "decomposition/gate_style_identity_error": (
            gate["gate_style_identity_error"]
        ),
        "decomposition/required_w_at_current_p_c": (
            gate["required_w_at_current_p_c"] or -1.0
        ),
        "decomposition/required_p_at_current_c_w": (
            gate["required_p_at_current_c_w"] or -1.0
        ),
    })
    print("new decomposition:", json.dumps(gate, indent=2)[:4000])

    if ROBUST_GATE_SHARDS and ROBUST_GATE_SHARDS > GATE_SHARDS:
        robust_dir = REPO_DIR / f"eval_results_{CKPT_TAG}_gate_{ROBUST_GATE_SHARDS}"
        robust_cmd = [
            PYTHON, "diagnose_audio_bug.py",
            "--ckpt", str(CKPT_DIR),
            "--teacher-cache", str(STATE["cache"]),
            "--shards", str(ROBUST_GATE_SHARDS),
            "--out", str(robust_dir),
            "--device", "cuda:0" if torch.cuda.is_available() else "cpu",
            "--no-audio",
        ]
        run(robust_cmd, cwd=REPO_DIR)
        robust = json.loads((robust_dir / "diagnostic_report.json").read_text())["summary"]
        gate_run.log({f"robust_gate/{k}": float(v) for k, v in robust.items()})
        print(f"{ROBUST_GATE_SHARDS}-shard robustness recheck:", robust)
        # The 4-shard result remains the gate verdict.

    if LOG_ARTIFACTS:
        art = wandb.Artifact(f"v7-gate-{DATASET_CHOICE}", type="evaluation")
        for path in (NEW_GATE_DIR / "diagnostic_report.json",
                     NEW_CEIL_DIR / "ceiling_report.json"):
            if path.is_file():
                art.add_file(str(path))
        gate_run.log_artifact(art)
    if gate_run.url:
        STATE["wandb_urls"].append(gate_run.url)
    print("W&B gate:", gate_run.url or gate_run.dir)
    gate_run.finish()
else:
    print("RUN_GATE=False; gate skipped")


## 9. 60-utterance evaluation — only after a real gate pass

This cell is off by default. It refuses to run if the gate failed. The gate is
not a formality: B/C quality numbers are not comparable when `rd_a < 5%`.


In [ ]:
if RUN_EVAL:
    if not RUN_GATE and not ALLOW_EVAL_WITHOUT_GATE:
        raise RuntimeError("RUN_EVAL requires RUN_GATE=True so the 5% gate runs first")
    if STATE.get("gate_pass") is False:
        raise RuntimeError(
            "Gate failed; refusing to run the 60-utterance evaluation. "
            "Do not lower or reinterpret the gate."
        )
    eval_cmd = [
        PYTHON, "run_full_evaluation.py",
        "--ckpt", str(CKPT_DIR),
        "--configs", "B,C",
        "--sample-mode", "argmax",
        "--results", str(EVAL_DIR / "raw_results.jsonl"),
        "--audio-dir", str(EVAL_AUDIO),
        "--figures-dir", str(EVAL_DIR),
        "--asr-model", ASR_MODEL,
        "--force",
        "--wandb",
        "--wandb-project", WANDB_PROJECT,
        "--wandb-group", WANDB_GROUP,
        "--wandb-run-name", f"v7-eval60-{DATASET_CHOICE}",
    ]
    run(eval_cmd, cwd=REPO_DIR)
    summary_path = EVAL_DIR / "summary.json"
    if summary_path.is_file():
        print(summary_path.read_text(encoding="utf-8")[:5000])
    else:
        print("WARNING: evaluation wrote no summary.json")
else:
    print("RUN_EVAL=False; 60-utterance evaluation skipped")


## 10. Final truth, W&B import, and bundle

The summary below is deliberately blunt. It prints the exact old and new
decompositions, whether the unchanged gate passed, and what the next honest
step is. It never converts a failure into a quality claim.


In [ ]:
print("=" * 80)
print("CodeSwitch-TTS v7 text/confidence residual — final truth")
print("  git commit :", STATE["commit"])
print("  base ckpt  :", STATE["base_ckpt"])
print("  cache      :", STATE["cache"])
print("  new ckpt   :", CKPT_DIR if RUN_TRAIN else "not trained")
base_meta = STATE.get("base_meta") or {}
print("  corpus     :", base_meta.get("corpus_hours"), "h |",
      base_meta.get("corpus_size"), "shards")
print("  prior W&B  :", PRIOR_RUNS)
print("-" * 80)

base_gate = STATE.get("base_gate")
new_gate = STATE.get("new_gate")
if base_gate:
    print("base gate  : rt_a={:.4f}% rd_a={:.4f}% ({})".format(
        base_gate["rt_a"], base_gate["rd_a"],
        "PASS" if base_gate["rt_a"] >= 5 and base_gate["rd_a"] >= 5 else "FAIL"))
if new_gate:
    print("new gate   : rt_a={:.4f}% rd_a={:.4f}% ({})".format(
        new_gate["rt_a"], new_gate["rd_a"],
        "PASS" if STATE.get("gate_pass") else "FAIL"))
    print("  cb0_s={:.4f}% cb0_a={:.4f}% rt_s={:.4f}% rd_s={:.4f}%".format(
        new_gate["cb0_s"], new_gate["cb0_a"], new_gate["rt_s"], new_gate["rd_s"]))

new_ceiling = STATE.get("new_ceiling") or {}
new_decomp = new_ceiling.get("gate") or {}
if new_decomp:
    p, c, w = new_decomp["p"], new_decomp["c"], new_decomp["w"]
    print("decomposition (frame-weighted): p={:.4f}% c={:.4f}% w={:.4f}% "
          "-> rd_a={:.4f}%".format(p, c, w, new_decomp["rd_a_from_terms"]))
    print("  official gate-style rd_a : {:.4f}% (equal-weight per shard)".format(
        new_decomp["rd_a_gate_style"]))
    print("  gate-style identity error: {:.4f} points (p/c/w means are "
          "nonlinear; not exact)".format(
              new_decomp["gate_style_identity_error"]))
    print("  text-only fallback w_text={}".format(
        f"{new_decomp['w_text_model']:.4f}%" if new_decomp.get("w_text_model") is not None else "n/a"))
    if new_decomp.get("required_w_at_current_p_c") is not None:
        print("  required w at current p/c: {:.2f}%".format(
            new_decomp["required_w_at_current_p_c"]))
    print("  truth:", new_ceiling.get("truth"))

print("-" * 80)
if STATE.get("gate_pass") is True:
    print("VERDICT: PASS. The unchanged rt_a/rd_a gate passed on this sample.")
    print("Next: run the 60-utterance evaluation, then report B/C quality with its")
    print("exact provenance. Do not generalize beyond the evaluated sample.")
elif STATE.get("gate_pass") is False:
    print("VERDICT: FAIL. The unchanged 5% gate did not pass.")
    print("No B/C quality claim. No 60-utterance quality claim.")
    print("The result is evidence about the tested intervention, not permission to")
    print("re-scope the gate. A metric change needs explicit user sign-off.")
else:
    print("VERDICT: INCOMPLETE. The gate was not run in this session.")
print("=" * 80)

if STATE["wandb_urls"]:
    print("W&B runs:")
    for url in STATE["wandb_urls"]:
        print("  ", url)
else:
    print("W&B: no online URLs recorded (offline mode?)")

if HF_PUSH_REPO and (CKPT_DIR / "residual_cb1_8.pt").is_file():
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ.get("HF_TOKEN"))
    api.create_repo(repo_id=HF_PUSH_REPO, repo_type="model", exist_ok=True, private=True)
    api.upload_folder(folder_path=str(CKPT_DIR), repo_id=HF_PUSH_REPO, repo_type="model")
    print("HF checkpoint pushed:", HF_PUSH_REPO)


In [ ]:
if DO_ZIP:
    bundle_root = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else REPO_DIR.parent
    bundle_path = bundle_root / f"codeswitch_{CKPT_TAG}_{DATASET_CHOICE}.zip"
    include = [CKPT_DIR, DIAG_DIR, BASE_GATE_DIR, BASE_CEIL_DIR, NEW_GATE_DIR,
               NEW_CEIL_DIR, EVAL_DIR]
    with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for item in include:
            if not item.exists():
                continue
            if item.is_dir():
                for child in sorted(item.rglob("*")):
                    if child.is_file():
                        archive.write(child, child.relative_to(REPO_DIR))
            else:
                archive.write(item, item.relative_to(REPO_DIR))
    print("bundle:", bundle_path, f"({bundle_path.stat().st_size / 1e6:.1f} MB)")
else:
    print("DO_ZIP=False; no bundle written")
